# <Project Name>

| Field | Value |
|-------|-------|
| **Authors** | Kayla Hooper & Diya Gumaste |
| **Course** | ADS 5834 — Data Science in Practice |
| **Notebook** | model_evaluation |
| **Created Date** | 2026-11-19 |
| **Last Updated Date** | 2026-11-19 |
| **Last Updated By** | Kayla Hooper |
| **Technical Owner Email** | khooper5@vt.edu |
| **Business Owner Email** | khooper5@vt.edu |

## Project Description


## Notebook Description

In this notebook, we evaluate the chosen model by scoring held-out test set with the registered champions, choosing EUO thresholds on validation by expected-dollar benefit, quantifty improved and recovery KPIs vs. a no-model baseline and random review, audit fairness and false-positive rate, and generate the EUO rationale per claim. For deployment design, we use a REST scoring contract.

In [0]:
SRC_TABLE = "workspace.insurance_fraud.auto_insurance_fraud"   # READ-ONLY warehouse table
DM = "workspace.dm_final_project"                               # ALL derived work goes here
SEED = 42
assert "insurance_fraud" not in DM, "Never write to the warehouse schema"

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, mlflow, mlflow.sklearn, json
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_recall_curve, roc_curve, confusion_matrix,
                             precision_score, recall_score, f1_score, brier_score_loss)
from sklearn.calibration import calibration_curve
sns.set_theme(style="whitegrid")
mlflow.set_registry_uri("databricks-uc")

# ---- BUSINESS ASSUMPTIONS (edit) -----------------------------------------------
ANNUAL_CLAIMS        = 150_000
EUO_COST             = 1_500    # investigator + counsel + court reporter per EUO (USD)
AVOIDED_RATE         = 0.60     # share of damage_amount avoided when an EUO is held on a truly fraudulent claim
FP_FRICTION_COST     = 250      # customer-relations / complaint-handling cost when a legitimate claimant is put through an EUO
REVIEW_CAPACITY_PCT  = 0.03     # max share of claims the SIU can examine
# ---------------------------------------------------------------------------------
TARGETS = {"organized": "is_organized_fraud", "opportunistic": "is_opportunistic_fraud"}
feat = spark.table(f"{DM}.features_all").toPandas()
FEATURES = spark.table(f"{DM}.feature_catalog").toPandas()["feature"].tolist()
va, te = feat[feat.split == "val"].copy(), feat[feat.split == "test"].copy()
client = mlflow.MlflowClient()
models, versions = {}, {}
for t in TARGETS:
    uri = f"models:/{DM}.fraud_{t}_model@champion"
    models[t] = mlflow.sklearn.load_model(uri)
    versions[t] = client.get_model_version_by_alias(f"{DM}.fraud_{t}_model", "champion").version
    va[f"score_{t}"] = models[t].predict_proba(va[FEATURES].astype(float))[:, 1]
    te[f"score_{t}"] = models[t].predict_proba(te[FEATURES].astype(float))[:, 1]
print("champion versions:", versions, "| test rows:", len(te))

## 1. Threshold selection on validation (maximize expected net benefit)
Net benefit = avoided payouts on caught fraud - EUO cost on everything flagged - friction cost on false positives. A capacity cap is also enforced.

In [0]:
def net_benefit(y, flag, damage):
    tp = (flag == 1) & (y == 1); fp = (flag == 1) & (y == 0)
    return (damage[tp].sum() * AVOIDED_RATE) - (flag.sum() * EUO_COST) - (fp.sum() * FP_FRICTION_COST)

THRESH = {}
curves = {}
for t, col in TARGETS.items():
    y, s, dmg = va[col].values, va[f"score_{t}"].values, va["ctx_damage_amount"].fillna(0).values
    grid = np.unique(np.quantile(s, np.linspace(0.80, 0.9995, 200)))
    nb = [net_benefit(y, (s >= g).astype(int), dmg) for g in grid]
    cap_ok = [((s >= g).mean() <= REVIEW_CAPACITY_PCT) for g in grid]
    nb_cap = [v if ok else -np.inf for v, ok in zip(nb, cap_ok)]
    i = int(np.argmax(nb_cap)); THRESH[t] = float(grid[i]); curves[t] = (grid, nb)
    print(f"{t:14s} threshold={grid[i]:.3f} | flag rate on val={(s >= grid[i]).mean():.2%} | val net benefit=${nb_cap[i]:,.0f}")
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, t in zip(axes, TARGETS):
    ax.plot(*curves[t]); ax.axvline(THRESH[t], color="crimson", ls="--"); ax.set_title(f"{t}: validation net benefit vs threshold"); ax.set_xlabel("score threshold"); ax.set_ylabel("USD")
plt.tight_layout(); plt.show()

## 2. Test-set performance (never used for tuning)

In [0]:
rows = []
for t, col in TARGETS.items():
    y, s = te[col].values, te[f"score_{t}"].values; flag = (s >= THRESH[t]).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, flag).ravel()
    rows.append(dict(model=t, prevalence=y.mean(), roc_auc=roc_auc_score(y, s), pr_auc=average_precision_score(y, s),
                     pr_auc_lift_vs_random=average_precision_score(y, s) / y.mean(), threshold=THRESH[t], flag_rate=flag.mean(),
                     precision=precision_score(y, flag, zero_division=0), recall=recall_score(y, flag), f1=f1_score(y, flag),
                     false_positive_rate=fp / (fp + tn), brier=brier_score_loss(y, s), TP=tp, FP=fp, FN=fn, TN=tn))
perf = pd.DataFrame(rows); display(perf.round(4))
spark.createDataFrame(perf).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DM}.test_performance")

In [0]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for r, (t, col) in enumerate(TARGETS.items()):
    y, s = te[col].values, te[f"score_{t}"].values
    p, rc, _ = precision_recall_curve(y, s); axes[r, 0].plot(rc, p); axes[r, 0].axhline(y.mean(), ls=":", c="gray"); axes[r, 0].set(title=f"{t}: precision-recall", xlabel="recall", ylabel="precision")
    fpr, tpr, _ = roc_curve(y, s); axes[r, 1].plot(fpr, tpr); axes[r, 1].plot([0, 1], [0, 1], ls=":", c="gray"); axes[r, 1].set(title=f"{t}: ROC", xlabel="FPR", ylabel="TPR")
    pt, pp = calibration_curve(y, s, n_bins=10, strategy="quantile"); axes[r, 2].plot(pp, pt, marker="o"); axes[r, 2].plot([0, 1], [0, 1], ls=":", c="gray"); axes[r, 2].set(title=f"{t}: calibration", xlabel="mean predicted", ylabel="observed")
plt.tight_layout(); plt.show()

# Gains / lift table by score decile
for t, col in TARGETS.items():
    d = te[[col, f"score_{t}"]].copy(); d["decile"] = pd.qcut(d[f"score_{t}"].rank(method="first", ascending=False), 10, labels=range(1, 11))
    g = d.groupby("decile")[col].agg(claims="count", frauds="sum", rate="mean"); g["cum_capture"] = g["frauds"].cumsum() / g["frauds"].sum(); g["lift"] = g["rate"] / d[col].mean()
    print(f"\n{t}: gains table"); display(g.round(3))

## 3. KPIs improved & recovery
Compares (a) no-model baseline (no EUOs, all fraud paid), (b) random EUO selection at the same volume, (c) our models. Dollar figures are scaled to IronKey's annual claim volume.

In [0]:
# Combined EUO decision: either model flags; priority = max of normalized scores
te["flag_org"] = (te["score_organized"] >= THRESH["organized"]).astype(int)
te["flag_opp"] = (te["score_opportunistic"] >= THRESH["opportunistic"]).astype(int)
te["euo_recommended"] = ((te.flag_org + te.flag_opp) > 0).astype(int)
te["any_fraud"] = ((te.is_organized_fraud + te.is_opportunistic_fraud) > 0).astype(int)
te["priority_score"] = np.maximum(te["score_organized"] / THRESH["organized"], te["score_opportunistic"] / THRESH["opportunistic"])

dmg = te["ctx_damage_amount"].fillna(0).values; y = te["any_fraud"].values
scale = ANNUAL_CLAIMS / len(te)
def scenario(flag):
    tp = (flag == 1) & (y == 1); fp = (flag == 1) & (y == 0)
    return dict(euo_volume=int(flag.sum()), euo_rate=flag.mean(), fraud_caught=int(tp.sum()), fraud_detection_rate=tp.sum() / max(y.sum(), 1),
                precision=tp.sum() / max(flag.sum(), 1), legit_claimants_flagged=int(fp.sum()), false_positive_rate=fp.sum() / max((y == 0).sum(), 1),
                fraud_payout_avoided=dmg[tp].sum() * AVOIDED_RATE, euo_spend=flag.sum() * EUO_COST, friction_cost=fp.sum() * FP_FRICTION_COST)
rng = np.random.default_rng(SEED)
flag_model = te["euo_recommended"].values
def random_flag():
    f = np.zeros(len(te), int); f[rng.choice(len(te), int(flag_model.sum()), replace=False)] = 1
    return f
rand_runs = [scenario(random_flag()) for _ in range(50)]
flag_rand_avg = pd.DataFrame(rand_runs).mean()
kpi = pd.DataFrame({"No model": scenario(np.zeros(len(te), int)), "Random EUO (same volume)": flag_rand_avg.to_dict(), "Fraud models": scenario(flag_model)}).T
kpi["net_benefit_test"] = kpi.fraud_payout_avoided - kpi.euo_spend - kpi.friction_cost
kpi["net_benefit_annualized"] = kpi.net_benefit_test * scale
display(kpi.round(3))
spark.createDataFrame(kpi.reset_index().rename(columns={"index": "scenario"})).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DM}.kpi_recovery")

m, r = kpi.loc["Fraud models"], kpi.loc["Random EUO (same volume)"]
print(f"Fraud detection rate: {m.fraud_detection_rate:.1%} vs {r.fraud_detection_rate:.1%} random  ({m.fraud_detection_rate / max(r.fraud_detection_rate, 1e-9):.1f}x)")
print(f"EUO precision:        {m.precision:.1%} vs {r.precision:.1%} random")
print(f"Estimated annual payout avoided: ${m.fraud_payout_avoided * scale:,.0f}  | annual net benefit: ${m.net_benefit_annualized:,.0f}  (vs ${r.net_benefit_annualized:,.0f} random)")

In [0]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
kpi[["fraud_detection_rate", "precision"]].plot.bar(ax=ax[0], rot=15); ax[0].set_title("Detection rate & precision")
(kpi["net_benefit_annualized"] / 1e6).plot.bar(ax=ax[1], rot=15, color="seagreen"); ax[1].set_title("Annualized net benefit ($M)"); plt.tight_layout(); plt.show()

## 4. Fairness / customer-relations audit
False positives damage customer relations and invite regulatory scrutiny. Demographics were **not** model inputs; here we verify the outcomes are not disproportionately concentrated on a group.

In [0]:
te["age_band"] = pd.cut(te["ctx_insured_age"], [0, 25, 35, 50, 65, 120], labels=["<=25", "26-35", "36-50", "51-65", "66+"])
def audit(group_col):
    g = te.groupby(group_col, observed=True).apply(lambda d: pd.Series(dict(
        claims=len(d), fraud_rate=d.any_fraud.mean(), flag_rate=d.euo_recommended.mean(),
        recall=(d.euo_recommended[d.any_fraud == 1].mean() if (d.any_fraud == 1).any() else np.nan),
        false_positive_rate=d.euo_recommended[d.any_fraud == 0].mean())))
    g["fpr_vs_overall"] = g["false_positive_rate"] / te.euo_recommended[te.any_fraud == 0].mean()
    return g
for col in ["ctx_state", "age_band"]:
    a = audit(col); print(f"\nBy {col}"); display(a.round(3))
    flagged_disp = a[(a.claims >= 30) & (a.fpr_vs_overall > 1.25)]
    if len(flagged_disp): print("!! False-positive rate >1.25x overall for:", flagged_disp.index.tolist())

## 5. EUO rationale generator (regulatory explainability)
Every EUO recommendation gets (a) the claim-specific evidence values, (b) the top contributing indicators, (c) the model version and threshold. Works for the logistic-regression champion (coefficient x standardized value) and the decision-tree champion (the exact rule path).

In [0]:
TEXT = {  # feature -> (template, "direction": a function deciding whether the value is noteworthy)
 "report_delay_hours":        "claim reported {v:.0f} hours after the incident",
 "log_report_delay":          "unusually delayed report of the incident",
 "policy_inception_days":     "policy was only {v:.0f} days old at the time of loss",
 "new_policy_30d":            "loss occurred within 30 days of policy inception",
 "damage_inflation_ratio":    "damage estimate grew to {v:.2f}x the initial estimate",
 "damage_inflation_abs":      "damage estimate increased by ${v:,.0f} after the initial report",
 "damage_inflated_25pct":     "damage estimate grew by 25% or more",
 "claim_amendments":          "claim was amended {v:.0f} time(s)",
 "settlement_inquiries":      "claimant/representative inquired about settlement amounts {v:.0f} time(s)",
 "medical_visits_count":      "{v:.0f} medical visits recorded",
 "days_to_attorney":          "attorney retained {v:.0f} day(s) after the incident",
 "same_day_attorney":         "attorney retained within one day of the incident",
 "num_people_in_car":         "{v:.0f} occupants in the vehicle",
 "surname_ratio":             "occupants share few surnames (distinct-surname ratio {v:.2f})",
 "unrelated_group":           "three or more apparently unrelated occupants",
 "provider_org_rate":         "medical provider has a historical organized-fraud rate of {v:.1%}",
 "attorney_org_rate":         "attorney firm has a historical organized-fraud rate of {v:.1%}",
 "provider_opp_rate":         "medical provider has a historical opportunistic-fraud rate of {v:.1%}",
 "attorney_opp_rate":         "attorney firm has a historical opportunistic-fraud rate of {v:.1%}",
 "nearby_claims_30d":         "{v:.0f} other claims within 1 km in the prior 30 days",
 "nearby_same_provider_30d":  "{v:.0f} nearby recent claims used the same medical provider",
 "nearby_same_attorney_30d":  "{v:.0f} nearby recent claims used the same attorney",
 "prior_claims_6mo":          "{v:.0f} prior claim(s) in the last 6 months",
 "night_incident":            "incident occurred overnight",
 "has_attorney":              "claimant is legally represented",
}
def phrase(f, v):
    if f.startswith("txt_"):  return f"adjuster notes show pattern '{f[4:].replace('_', ' ')}'"
    if f.startswith("ng_"):   return f"adjuster notes contain the phrase '{f[3:].replace('_', ' ')}'"
    return TEXT.get(f, f"{f} = {v:.2f}").format(v=v)

def explain_claim(model, x_row, k=4):
    """x_row: 1-row DataFrame of FEATURES. Returns list of (feature, value, contribution, sentence)."""
    clf = model.named_steps["clf"]
    if hasattr(clf, "coef_"):
        Xs = model[:-1].transform(x_row)[0]
        contrib = clf.coef_[0] * Xs
        order = np.argsort(-contrib)[:k]
        return [(FEATURES[i], float(x_row.iloc[0, i]), float(contrib[i]), phrase(FEATURES[i], float(x_row.iloc[0, i]))) for i in order if contrib[i] > 0]
    # decision tree: exact rule path
    Xi = model.named_steps["imp"].transform(x_row); tree = clf.tree_
    out = []
    for n in clf.decision_path(Xi).indices:
        if tree.children_left[n] == -1: continue
        f_i, thr, v = tree.feature[n], tree.threshold[n], Xi[0, tree.feature[n]]
        out.append((FEATURES[f_i], float(v), float(thr), f"{phrase(FEATURES[f_i], float(v))} (rule: {FEATURES[f_i]} {'<=' if v <= thr else '>'} {thr:.2f})"))
    return out[:k + 2]

def build_rationale(row, x_row):
    parts, feats = [], {}
    for t in TARGETS:
        if row[f"flag_{'org' if t == 'organized' else 'opp'}"] == 1:
            ex = explain_claim(models[t], x_row)
            parts.append(f"{t.capitalize()}-fraud indicators (score {row['score_' + t]:.2f} >= threshold {THRESH[t]:.2f}; model v{versions[t]}): " + "; ".join(e[3] for e in ex) + ".")
            feats[t] = [dict(feature=e[0], value=e[1], contribution_or_threshold=e[2]) for e in ex]
    return " ".join(parts), feats

In [0]:
flagged = te[te.euo_recommended == 1].sort_values("priority_score", ascending=False)
rat, feats_json = [], []
for _, row in flagged.iterrows():
    text, feats = build_rationale(row, row[FEATURES].astype(float).to_frame().T)
    rat.append(f"Claim {row.claim_id} recommended for Examination Under Oath. " + text); feats_json.append(json.dumps(feats))
flagged = flagged.assign(rationale=rat, top_features_json=feats_json)
for r in flagged.rationale.head(5): print("-", r, "\n")

In [0]:
# Persist the audit trail (one row per scored test claim; every decision is reproducible)
audit_df = te.assign(rationale="", top_features_json="").set_index("claim_id")
audit_df.loc[flagged.claim_id, "rationale"] = flagged.set_index("claim_id")["rationale"]
audit_df.loc[flagged.claim_id, "top_features_json"] = flagged.set_index("claim_id")["top_features_json"]
audit_out = audit_df.reset_index()[["claim_id", "score_organized", "score_opportunistic", "flag_org", "flag_opp", "euo_recommended", "priority_score", "rationale", "top_features_json"]].copy()
audit_out["threshold_organized"] = THRESH["organized"]; audit_out["threshold_opportunistic"] = THRESH["opportunistic"]
audit_out["model_version_organized"] = str(versions["organized"]); audit_out["model_version_opportunistic"] = str(versions["opportunistic"])
audit_out["scored_at"] = pd.Timestamp.now()
spark.createDataFrame(audit_out).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DM}.euo_audit_trail")
print(f"{len(audit_out):,} decisions logged; {int(audit_out.euo_recommended.sum()):,} EUO recommendations")

## 6. Deployment into a claims management system via REST (design-doc material)
**Pattern:** the claims system calls a scoring endpoint at FNOL and again whenever the claim is amended/adds notes.

```
Claims system --(POST /v1/claims/{claim_id}/score)--> Scoring API --> feature service --> champion models --> rationale builder
                                                                          |-> provider_rate_lookup (UC table, Delta)
                                                                          |-> recent-claims geo index (30-day window)
                                                       --> euo_audit_trail (append-only)
```

**Request**
```json
{"claim_id": "CLM000123", "claim_date": "2026-11-17", "state": "VA", "num_people_in_car": 4, "num_different_surnames": 4,
 "report_delay_hours": 3, "medical_provider": "ADVANCED PAIN INSTITUTE", "attorney_firm": "MAXIMUM COMPENSATION LEGAL",
 "incident_latitude": 37.23, "incident_longitude": -80.41, "damage_amount": 5800.0, "initial_damage_amount": 3200.0,
 "claim_amendments": 2, "medical_visits_count": 4, "days_to_attorney": 0, "settlement_inquiries": 2, "claim_notes": "..."}
```
**Response**
```json
{"claim_id": "CLM000123", "euo_recommended": true, "priority_score": 2.4,
 "scores": {"organized": 0.91, "opportunistic": 0.12}, "thresholds": {"organized": 0.42, "opportunistic": 0.55},
 "rationale": "Organized-fraud indicators ...", "top_features": [...], "model_versions": {"organized": "3", "opportunistic": "2"},
 "decision_id": "uuid", "scored_at": "2026-11-17T14:53:00Z"}
```
**Serving notes:** register the feature-engineering code (notebook 03 functions) with the model as a pyfunc wrapper or a feature-serving endpoint; models are loaded by the `@champion` alias so promotion = alias move (with rollback); log every request/response to `euo_audit_trail`; monitor score drift, flag rate and false-positive rate monthly; humans make the final EUO decision.

A Databricks Model Serving call looks like:
```python
requests.post(f"{HOST}/serving-endpoints/fraud_organized/invocations", headers={"Authorization": f"Bearer {TOKEN}"},
              json={"dataframe_split": {"columns": FEATURES, "data": [[...]]}})
```

## 7. Summary for the presentation
Fill in from the cells above: detection rate vs. random, precision, annualized payout avoided and net benefit, false-positive rate by group, interpretability cost (champion vs. challenger), and the assumptions that drive the dollar figures.